# Dashboard notebook: food and housing insecurity monitoring

This notebook builds a compact dashboard using the synthetic dataset. It summarizes food insecurity, housing insecurity, hardship score, and policy-priority segments. The code also exports a standalone HTML dashboard.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Create key indicators

These indicators are suitable for a high-level policy or program dashboard. They should be read as synthetic examples, not real population estimates.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Food insecure rate",
        "Housing insecure rate",
        "Average hardship score",
        "Very low food security rate",
    ],
    "value": [
        f"{len(df):,}",
        f"{df['food_insecure_label'].mean():.1%}",
        f"{df['housing_insecure_label'].mean():.1%}",
        f"{df['overall_hardship_score'].mean():.1f}",
        f"{(df['food_security_status'].eq('very_low_food_security')).mean():.1%}",
    ],
})
summary

## 2. Group-level rate tables

The helper below calculates rates by any selected grouping column.

In [ ]:
def rate_table(group_col: str) -> pd.DataFrame:
    out = (
        df.groupby(group_col, dropna=False)
        .agg(
            households=("household_id", "count"),
            food_insecure_rate=("food_insecure_label", "mean"),
            housing_insecure_rate=("housing_insecure_label", "mean"),
            avg_hardship_score=("overall_hardship_score", "mean"),
        )
        .reset_index()
        .sort_values("households", ascending=False)
    )
    return out

rate_table("region")

## 3. Interactive charts

These Plotly charts can run inside Jupyter. They are also exported to a single HTML file in the next section.

In [ ]:
import plotly.express as px

segment_counts = (
    df["policy_priority_segment"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("policy_priority_segment")
    .reset_index(name="percent")
)

fig_segments = px.bar(
    segment_counts,
    x="policy_priority_segment",
    y="percent",
    title="Policy-priority segment distribution",
    labels={"percent": "Percent of households", "policy_priority_segment": "Segment"},
)
fig_segments.update_layout(xaxis_tickangle=-30)
fig_segments.show()

In [ ]:
region_rates = rate_table("region")
fig_region = px.bar(
    region_rates,
    x="region",
    y=["food_insecure_rate", "housing_insecure_rate"],
    barmode="group",
    title="Food and housing insecurity rates by region",
    labels={"value": "Rate", "variable": "Measure"},
)
fig_region.show()

In [ ]:
fig_hardship = px.histogram(
    df,
    x="overall_hardship_score",
    nbins=40,
    color="food_insecure_label",
    title="Hardship score distribution by food insecurity label",
    labels={"overall_hardship_score": "Hardship score", "food_insecure_label": "Food insecure"},
)
fig_hardship.show()

In [ ]:
state_rates = (
    df.groupby("state")
    .agg(
        households=("household_id", "count"),
        food_insecure_rate=("food_insecure_label", "mean"),
        housing_insecure_rate=("housing_insecure_label", "mean"),
        avg_hardship_score=("overall_hardship_score", "mean"),
    )
    .reset_index()
)

fig_state = px.choropleth(
    state_rates,
    locations="state",
    locationmode="USA-states",
    color="food_insecure_rate",
    scope="usa",
    hover_data=["households", "housing_insecure_rate", "avg_hardship_score"],
    title="Synthetic food insecurity rate by state",
)
fig_state.show()

## 4. Export a standalone HTML dashboard

The generated HTML file can be opened in a browser without running a notebook server.

In [ ]:
from pathlib import Path
import plotly.io as pio

kpi_html = summary.to_html(index=False, escape=False)
figures = [fig_segments, fig_region, fig_hardship, fig_state]
fig_html = "\n".join(
    pio.to_html(fig, include_plotlyjs=(i == 0), full_html=False)
    for i, fig in enumerate(figures)
)

html = f"""
<html>
<head>
  <meta charset="utf-8">
  <title>Synthetic Food and Housing Insecurity Dashboard</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 32px; }}
    h1 {{ margin-bottom: 6px; }}
    .note {{ color: #555; max-width: 900px; }}
    table {{ border-collapse: collapse; margin: 18px 0; }}
    th, td {{ border: 1px solid #ddd; padding: 8px 12px; }}
    th {{ background: #f2f2f2; }}
  </style>
</head>
<body>
  <h1>Synthetic Food and Housing Insecurity Dashboard</h1>
  <p class="note">This dashboard uses a fully synthetic dataset. The rates are for modeling practice and should not be interpreted as real survey estimates.</p>
  <h2>Key indicators</h2>
  {kpi_html}
  {fig_html}
</body>
</html>
"""

html_path = Path("food_housing_insecurity_dashboard.html")
html_path.write_text(html, encoding="utf-8")
html_path

## 5. Optional Streamlit app code

Run the exported Python file with `streamlit run food_housing_streamlit_app.py` from the folder that contains the dataset.

In [ ]:
streamlit_code = r"""
from pathlib import Path
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Food and Housing Insecurity Dashboard", layout="wide")
st.title("Synthetic Food and Housing Insecurity Dashboard")
st.caption("Fully synthetic data for practice. Do not use these rates as real estimates.")

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")
df = pd.read_csv(DATA_PATH)

region = st.sidebar.multiselect("Region", sorted(df["region"].unique()), default=sorted(df["region"].unique()))
segment = st.sidebar.multiselect("Policy segment", sorted(df["policy_priority_segment"].unique()), default=sorted(df["policy_priority_segment"].unique()))
view = df[df["region"].isin(region) & df["policy_priority_segment"].isin(segment)]

c1, c2, c3, c4 = st.columns(4)
c1.metric("Households", f"{len(view):,}")
c2.metric("Food insecure", f"{view['food_insecure_label'].mean():.1%}")
c3.metric("Housing insecure", f"{view['housing_insecure_label'].mean():.1%}")
c4.metric("Avg. hardship", f"{view['overall_hardship_score'].mean():.1f}")

segment_counts = view["policy_priority_segment"].value_counts(normalize=True).mul(100).reset_index()
segment_counts.columns = ["policy_priority_segment", "percent"]
st.plotly_chart(px.bar(segment_counts, x="policy_priority_segment", y="percent", title="Policy segment distribution"), use_container_width=True)

region_rates = view.groupby("region").agg(food_insecure_rate=("food_insecure_label", "mean"), housing_insecure_rate=("housing_insecure_label", "mean")).reset_index()
st.plotly_chart(px.bar(region_rates, x="region", y=["food_insecure_rate", "housing_insecure_rate"], barmode="group", title="Insecurity rates by region"), use_container_width=True)

st.dataframe(view.head(1000), use_container_width=True)
"""

app_path = Path("food_housing_streamlit_app.py")
app_path.write_text(streamlit_code, encoding="utf-8")
app_path